
# SWYNEX | End-to-End Employee Attrition Prediction

## Project Overview
An end-to-end machine learning workflow for exploring employee attrition data, preparing features, training and evaluating classification models, and generating predictions.

## Objectives
- Validate and explore the dataset.
- Build leakage-safe preprocessing pipelines.
- Train and compare classification models.
- Evaluate performance using multiple metrics.
- Save and reload the complete model pipeline.
- Generate predictions for new records.

## Technology Stack
Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, Joblib.

## Responsible Use
This is an educational predictive analytics project. Predictions are uncertain and should not be used as the sole basis for decisions affecting employees.


In [ ]:
##Step 2. Install and import the libraries

%pip install -q pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import os
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42

print("Environment initialized successfully.")
print("Python libraries loaded.")
print("Scikit-learn version:", sklearn.__version__)

In [ ]:
##Step 3. Upload your Employee Attrition CSV

from google.colab import files

uploaded = files.upload()

csv_files = [
    filename for filename in uploaded
    if filename.lower().endswith(".csv")
]

if not csv_files:
    raise FileNotFoundError(
        "No CSV uploaded. Please upload the Employee Attrition CSV."
    )

DATA_PATH = csv_files[0]
df = pd.read_csv(DATA_PATH)

print("Dataset loaded:", DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
##Step 4. Validate the dataset

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values per column:")
display(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Missing Values")
)

print("\nDuplicate rows:", df.duplicated().sum())

print("\nAttrition distribution:")
if "Attrition" in df.columns:
    display(df["Attrition"].value_counts(dropna=False).to_frame("Count"))
    display(
        (df["Attrition"].value_counts(normalize=True) * 100)
        .round(2)
        .to_frame("Percentage")
    )
else:
    raise ValueError("Required target column 'Attrition' was not found.")

In [ ]:
##Step 5. Create the first visualization

sns.set_theme(style="whitegrid")

attrition_counts = df["Attrition"].value_counts()

plt.figure(figsize=(7, 5))
sns.barplot(
    x=attrition_counts.index,
    y=attrition_counts.values
)
plt.title("Employee Attrition Distribution")
plt.xlabel("Attrition")
plt.ylabel("Number of Employees")
plt.tight_layout()
plt.show()

Phase 2 — Build leakage-safe preprocessing

In [ ]:
##Step 6. Define the prediction target and remove unsuitable columns

TARGET = "Attrition"

if TARGET not in df.columns:
    raise ValueError(f"Missing target column: {TARGET}")

data = df.copy()

# Remove rows without a known target.
data = data.dropna(subset=[TARGET]).copy()

# Keep only the expected binary classification labels.
data[TARGET] = data[TARGET].astype(str).str.strip()
data = data[data[TARGET].isin(["Yes", "No"])].copy()

# Separate target and features.
y = data[TARGET].map({"No": 0, "Yes": 1})
X = data.drop(columns=[TARGET])

# Remove identifier columns.
identifier_columns = [
    col for col in ["EmployeeNumber", "EmployeeID", "EmployeeId"]
    if col in X.columns
]
X = X.drop(columns=identifier_columns)

# Remove columns that have only one non-missing value.
constant_columns = [
    col for col in X.columns
    if X[col].nunique(dropna=True) <= 1
]
X = X.drop(columns=constant_columns)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Identifier columns removed:", identifier_columns)
print("Constant columns removed:", constant_columns)
print("\nTarget distribution:")
print(y.value_counts().rename(index={0: "No", 1: "Yes"}))

In [ ]:
##Step 7. Split the dataset before fitting transformations

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(3))

In [ ]:
##Step 8. Create reusable preprocessing pipelines

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Preprocessing pipeline created.")

Phase 3 — Train and compare models

In [ ]:
##Step 9. Train Logistic Regression and Random Forest

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

trained_models = {}
results = []
predictions = {}

for name, classifier in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    trained_models[name] = pipeline
    predictions[name] = {
        "y_pred": y_pred,
        "y_prob": y_prob
    }

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

comparison_df = pd.DataFrame(results).sort_values(
    by="F1-Score", ascending=False
).reset_index(drop=True)

display(comparison_df.round(4))

In [ ]:
##Step 10. Visualize the comparison

metrics_to_plot = [
    "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"
]

comparison_long = comparison_df.melt(
    id_vars="Model",
    value_vars=metrics_to_plot,
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=comparison_long,
    x="Metric",
    y="Score",
    hue="Model"
)
plt.ylim(0, 1)
plt.title("Employee Attrition Model Comparison")
plt.tight_layout()
plt.show()

In [ ]:
##Step 11. Inspect detailed performance

for name, pipeline in trained_models.items():
    y_pred = predictions[name]["y_pred"]

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["No Attrition", "Attrition"],
        zero_division=0
    ))

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        labels=[0, 1],
        display_labels=["No Attrition", "Attrition"],
        values_format="d"
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()

In [ ]:
##Phase 4 — Improve the selected model

tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

param_grid = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10],
    "classifier__class_weight": [None, "balanced"]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    refit=True
)

grid_search.fit(X_train, y_train)

final_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Best mean cross-validation recall:", round(
    grid_search.best_score_, 4
))

In [ ]:
##Step 13. Evaluate the tuned model once on the test set

final_pred = final_model.predict(X_test)
final_prob = final_model.predict_proba(X_test)[:, 1]

final_metrics = {
    "model": "Tuned Logistic Regression",
    "test_samples": int(len(y_test)),
    "accuracy": float(accuracy_score(y_test, final_pred)),
    "precision": float(precision_score(
        y_test, final_pred, zero_division=0
    )),
    "recall": float(recall_score(
        y_test, final_pred, zero_division=0
    )),
    "f1_score": float(f1_score(
        y_test, final_pred, zero_division=0
    )),
    "roc_auc": float(roc_auc_score(y_test, final_prob)),
    "best_parameters": grid_search.best_params_
}

print(json.dumps(final_metrics, indent=2))

In [ ]:
RocCurveDisplay.from_predictions(y_test, final_prob)
plt.title("ROC Curve — Tuned Logistic Regression")
plt.tight_layout()
plt.show()

Phase 5 — Save the model and demonstrate predictions

In [ ]:
##Step 14. Create the output folders

PROJECT_DIR = Path("/content/SWYNEX-End-to-End-ML-Project")

for folder in [
    "models",
    "results",
    "results/figures",
    "src",
    "tests",
    "data"
]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project directories created:", PROJECT_DIR)

In [ ]:
##Step 15. Save the final pipeline and results

model_path = PROJECT_DIR / "models" / "employee_attrition_model.joblib"
metrics_path = PROJECT_DIR / "results" / "final_metrics.json"
comparison_path = PROJECT_DIR / "results" / "model_comparison.csv"

joblib.dump(final_model, model_path)

comparison_df.to_csv(comparison_path, index=False)

with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump(final_metrics, file, indent=2)

print("Model saved:", model_path)
print("Metrics saved:", metrics_path)
print("Comparison saved:", comparison_path)

In [ ]:
##Step 16. Test loading the saved model and predicting

loaded_model = joblib.load(model_path)

# Use a held-out test record as a demonstration input.
# This demonstrates the prediction interface; it is not new external data.
sample_employee = X_test.iloc[[0]].copy()

predicted_class = loaded_model.predict(sample_employee)[0]
predicted_probability = loaded_model.predict_proba(
    sample_employee
)[0, 1]

print("Predicted class:", "Yes" if predicted_class == 1 else "No")
print(f"Predicted probability of attrition: {predicted_probability:.2%}")

##Step 18. Organize the final repository

SWYNEX-End-to-End-ML-Project/
├── notebooks/
│   └── SWYNEX_End_to_End_Employee_Attrition.ipynb
├── src/
│   ├── train.py
│   ├── predict.py
│   └── evaluate.py
├── tests/
│   └── test_prediction.py
├── models/
│   └── README.md
├── results/
│   ├── model_comparison.csv
│   ├── final_metrics.json
│   └── figures/
├── data/
│   └── README.md
├── README.md
├── requirements.txt
└── .gitignore

##Step 19. Add requirements.txt


pandas
numpy
matplotlib
seaborn
scikit-learn
joblib
pytest


##Step 20. Add .gitignore


__pycache__/
*.py[cod]
.ipynb_checkpoints/
.env
.venv/
venv/
.DS_Store
*.log

# Raw datasets should not be committed by default
data/*.csv

# Large generated artifacts: publish separately if needed
# models/*.joblib


## Final step — Download generated model and results
After all previous cells run successfully, this cell creates a ZIP containing the trained model and evaluation outputs. Download it and copy the contents into the matching folders of your local project.

In [ ]:
import shutil
from google.colab import files

artifact_zip = "/content/SWYNEX-ML-Artifacts.zip"
shutil.make_archive(artifact_zip[:-4], "zip", root_dir=str(PROJECT_DIR))
print("Created artifact archive:", artifact_zip)
files.download(artifact_zip)
